STAGE 1 IMPLEMENTATION (Corpus + Extraction)

In [1]:
!pip install pdfplumber pymupdf

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.6 MB 3.4 MB/s eta 0:00:02
   ------- -------------------------------- 1.3/6.6 MB 3.7 MB/s eta 0:00:02
   --------------- ------------------------ 2.6/6.6 MB 4.2 MB/s eta 0:00:01
   ---------------------- ----------------- 3.7/6.6 MB 4.4 MB/s eta 0:00:01
   ------------------------------- -------- 5.2/6.6 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------  6.6/6.6 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 5.2 MB/s  0:00:01
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   --- ------------------------------------ 1.6/19.2 MB 8.4 MB/s eta 0:00:03
   ----- ---------------------------------- 2.9/19.2 MB 7.3 MB/s eta 0:00:03
   ---------- ----------------------------- 5.2/19.2 MB 8.2 MB/s eta 0:00:02
   -------------- ------------------------- 7.1/19.2 MB 8.4 MB/s eta 0:00:02
   ------------------- -

In [2]:
import pdfplumber
import os

In [5]:
#load pdf
pdf_path = "../data/raw/force_laws_motion.pdf"

In [6]:
#extract text from pdf
all_text = ""

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text:
            all_text += f"\n\n--- PAGE {i+1} ---\n\n"
            all_text += text

print("Extraction done")

Extraction done


In [7]:
print(all_text[:2000])



--- PAGE 1 ---

8
C
hapter
FFFFF LLLLL MMMMM
OOOOORRRRRCCCCCEEEEE AAAAANNNNNDDDDD AAAAAWWWWWSSSSS OOOOOFFFFF OOOOOTTTTTIIIIIOOOOONNNNN
In the previous chapter, we described the In our everyday life we observe that some
motion of an object along a straight line in effort is required to put a stationary object
terms of its position, velocity and acceleration. into motion or to stop a moving object. We
We saw that such a motion can be uniform ordinarily experience this as a muscular effort
or non-uniform. We have not yet discovered and say that we must push or hit or pull on
what causes the motion. Why does the speed an object to change its state of motion. The
of an object change with time? Do all motions concept of force is based on this push, hit or
require a cause? If so, what is the nature of pull. Let us now ponder about a ‘force’. What
this cause? In this chapter we shall make an is it? In fact, no one has seen, tasted or felt a
attempt to quench all such curiosities. force. Howe

In [44]:
#cleaning text for better processing
def advanced_clean_text(text):
    import re
    
    # Remove "Reprint" noise
    text = re.sub(r'Reprint \d{4}-\d{2}', ' ', text)
    
    # Remove standalone numbers (like page numbers)
    text = re.sub(r'\b\d+\b', ' ', text)
    
    # Remove empty brackets
    text = re.sub(r'\(\s*\)', ' ', text)
    # Remove page markers
    text = re.sub(r'--- PAGE \d+ ---', ' ', text)
    
    # Remove repeated uppercase noise (e.g. OOOOORRRRR)
    text = re.sub(r'\b[A-Z]{3,}\b', ' ', text)
    
    # Fix broken words like "C hapter"
    text = re.sub(r'\b([A-Za-z])\s+([a-z])\b', r'\1\2', text)
    
    # Remove figure references
    text = re.sub(r'Fig\.\s*\d+(\.\d+)?', ' ', text)
    
    # Remove (a), (b), etc.
    text = re.sub(r'\([a-z]\)', ' ', text)
    
    # Fix sentence spacing (add period space)
    text = re.sub(r'\.\s+', '. ', text)
    
    # Replace newlines with space
    text = re.sub(r'\n+', ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

In [45]:
#clened text not ideal but acceptable for now. We can always improve this later.
cleaned_text = advanced_clean_text(all_text)

print(cleaned_text[:2000])

--- --- C hapter In the previous chapter, we described the In our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. We We saw that such a motion can be uniform ordinarily experience this as a muscular effort or non-uniform. We have not yet discovered and say that we must push or hit or pull on what causes the motion. Why does the speed an object to change its state of motion. The of an object change with time? Do all motions concept of force is based on this push, hit or require a cause? If so, what is the nature of pull. Let us now ponder about a ‘force’. What this cause? In this chapter we shall make an is it? In fact, no one has seen, tasted or felt a attempt to quench all such curiosities. force. However, we always see or feel the effect For many centuries, the problem of of a force. It can only be explained by motion 

In [47]:
clean_path = "../data/processed/cleaned_text.txt"

with open(clean_path, "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print("Cleaned text saved")

Cleaned text saved


STAGE 1 — PART 2: STRUCTURE DETECTION

In [46]:
#Split into smaller units
def split_into_blocks(text, max_len=300):
    words = text.split()
    
    blocks = []
    current = []
    
    for word in words:
        current.append(word)
        
        if len(current) >= max_len:
            blocks.append(" ".join(current))
            current = []
    
    if current:
        blocks.append(" ".join(current))
    
    return blocks

blocks = split_into_blocks(cleaned_text)

print(len(blocks))
print(blocks[0][:500])

21
--- --- C hapter In the previous chapter, we described the In our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. We We saw that such a motion can be uniform ordinarily experience this as a muscular effort or non-uniform. We have not yet discovered and say that we must push or hit or pull on what causes the motion. Why does the sp


In [37]:
#Detect types (core logic)
#concept
#example
#question
def classify_block(text):
    t = text.lower()
    
    # Example detection (stronger)
    if "example" in t:
        return "example"
    
    # Question detection (better rule)
    if "?" in text and len(text) < 200:
        return "question"
    
    return "concept"

In [38]:
#Apply classification
structured_blocks = []

for block in blocks:
    structured_blocks.append({
        "text": block,
        "type": classify_block(block)
    })

structured_blocks[:5]

[{'text': '8 C hapter In the previous chapter, we described the In our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. We We saw that such a motion can be uniform ordinarily experience this as a muscular effort or non-uniform. We have not yet discovered and say that we must push or hit or pull on what causes the motion. Why does the speed an object to change its state of motion. The of an object change with time? Do all motions concept of force is based on this push, hit or require a cause? If so, what is the nature of pull. Let us now ponder about a ‘force’. What this cause? In this chapter we shall make an is it? In fact, no one has seen, tasted or felt a attempt to quench all such curiosities. force. However, we always see or feel the effect For many centuries, the problem of of a force. It can only be explained by mo

In [40]:

#Inspect classification
for item in structured_blocks[:20]:
    print(item)
    print("------")

{'text': '8 C hapter In the previous chapter, we described the In our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. We We saw that such a motion can be uniform ordinarily experience this as a muscular effort or non-uniform. We have not yet discovered and say that we must push or hit or pull on what causes the motion. Why does the speed an object to change its state of motion. The of an object change with time? Do all motions concept of force is based on this push, hit or require a cause? If so, what is the nature of pull. Let us now ponder about a ‘force’. What this cause? In this chapter we shall make an is it? In fact, no one has seen, tasted or felt a attempt to quench all such curiosities. force. However, we always see or feel the effect For many centuries, the problem of of a force. It can only be explained by mot

Key insight (this is important)

👉 we have reached:

✔️ “usable raw structure”
❌ “not retrieval-ready yet”

STEP 3 — CHUNKING WITH OVERLAP

In [41]:
def create_chunks(text, chunk_size=350, overlap=60):
    words = text.split()
    
    chunks = []
    start = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        
        chunk = " ".join(chunk_words)
        chunks.append(chunk)
        
        start = end - overlap
    
    return chunks

In [42]:
final_chunks = create_chunks(cleaned_text)

print(len(final_chunks))
print(final_chunks[0][:500])

22
8 C hapter In the previous chapter, we described the In our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. We We saw that such a motion can be uniform ordinarily experience this as a muscular effort or non-uniform. We have not yet discovered and say that we must push or hit or pull on what causes the motion. Why does the speed an


In [48]:
import json

chunk_data = []

for i, chunk in enumerate(final_chunks):
    chunk_data.append({
        "id": i,
        "text": chunk,
        "chapter": "Force and Laws of Motion"
    })

with open("../data/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunk_data, f, indent=4)

print("Chunks saved")

Chunks saved


In [49]:
with open("../data/processed/chunks.json", "r", encoding="utf-8") as f:
    test = json.load(f)

print(len(test))
print(test[0])

22
{'id': 0, 'text': '8 C hapter In the previous chapter, we described the In our everyday life we observe that some motion of an object along a straight line in effort is required to put a stationary object terms of its position, velocity and acceleration. into motion or to stop a moving object. We We saw that such a motion can be uniform ordinarily experience this as a muscular effort or non-uniform. We have not yet discovered and say that we must push or hit or pull on what causes the motion. Why does the speed an object to change its state of motion. The of an object change with time? Do all motions concept of force is based on this push, hit or require a cause? If so, what is the nature of pull. Let us now ponder about a ‘force’. What this cause? In this chapter we shall make an is it? In fact, no one has seen, tasted or felt a attempt to quench all such curiosities. force. However, we always see or feel the effect For many centuries, the problem of of a force. It can only be expl